# Sailor-7B-Chat Task 1 → Task 2 on Kaggle

Enable Internet and select **GPU T4 ×2** for default training. Choose `RUN_MODE`: `task1`, `task2`, or `both`. Fresh Task 1 with `GENERATE_OOF=True` needs all three private audited files: `VGEC 2026_v1.json`, `VGEC_classifier.json`, and `VGEC_balanced.json`. Task 2 with a reused classifier needs only the correction dataset and prior saved output. Task 1 without OOF needs only the original and classifier datasets, but cannot supply leakage-safe hints for later Task 2. Task 1 and Task 2 are distinct QLoRA adapters, not classifier-to-corrector weight transfer. Predicted class 0 never bypasses correction.

**Private GitHub access:** create a fine-grained token restricted to `rquq/GECC_VLSP_26` with **Contents: Read-only**. In Kaggle **Add-ons → Secrets**, add `GITHUB_TOKEN` and enable notebook access before Run All. Each bootstrap run clones the branch into a fresh source directory without putting the token in URLs or saved Git config. Old source edits and outputs are untouched. This does not download datasets: attach the private Kaggle Dataset separately.

**Weights & Biases:** for online tracking add `WANDB_API_KEY` in Kaggle **Add-ons → Secrets** and enable notebook access. The training subprocess inherits the key through its environment; it is never printed or saved in configs. No interactive `wandb login` is needed. `WANDB_MODE="offline"` or `TRACKING_ENABLED=False` avoids online logging. Main fits, OOF fits and diagnostics share one group. The Logs tab shows safe training/evaluation progress; the root results table updates after the classifier, each fold, aggregate OOF and corrector finish. W&B keeps compact outcomes and model artifacts, not repeated configs/parameter dumps; full reports remain local. Raw console capture is disabled, and corpus text/hints cannot be uploaded under the supplied agreement. Keep projects private. Private hosting alone is not permission to redistribute corpus-derived resources; obtain authorization before placing them on Kaggle.

`configs/baseline.json` uses pinned `sail/Sailor-7B-Chat`, NF4 QLoRA, rank 16 / alpha 32 / dropout 0.05, seed 42, five epochs, LR 5e-5 and effective batch eight. Attach the rebuilt corpus-derived datasets together; old OOF hints cannot be reused with changed data. A fresh fit selects its own best epoch by validation macro F1; historical scores are not guarantees.

The Task 1 baseline remains five epochs, LR 5e-5, effective batch eight, seed 42 and rank 16. Its five OOF fits use three epochs each and exclude connected source/parent/template groups. Task 2 retains the configured recipe (eight epochs, LR 2e-5), with optional localized edit targets. Select by error-only GLEU+, then exact match, then clean preservation. `HINT_ABLATION` and `CANDIDATE_RERANKING` evaluate the saved corrector after training; set `RUN_TRAINING=False` and `PRIOR_RUN` to evaluate an existing adapter without a new fit. These are experiments, not guaranteed improvements.

Both stages and every OOF fit use two-process DDP: one NF4 replica on each T4, synchronized adapter gradients, per-GPU microbatch one and four accumulation steps for global effective batch eight. T4 uses FP16. Epoch validation and generation run on rank 0; the other rank waits. Two GPUs are required by default; this is data parallelism for training throughput, not a split model or a promise of 2× speed.

Inputs are discovered under `/kaggle/input`; set explicit paths below if filenames are ambiguous. Attach `public_test.json` separately or with the three training JSONs: `SUBMISSION_DATA="auto"` discovers it for inference only, never training/OOF. Outputs live in `/kaggle/working/runs` for notebook-version retention. After Task 2, saved adapters infer the submission input into `RUN/submission.json`; no public-test scores are fabricated because labels are absent. Missing automatic input (or `SUBMISSION_DATA=None`) exports the labeled validation set after training. Keep datasets, notebook versions, checkpoints and text-containing outputs private.

**Save Version:** use **Save & Run All** for an unattended run. Training dependencies are installed in a separate Python environment that reuses Kaggle's CUDA PyTorch. The notebook kernel and its preloaded NumPy are untouched; no kernel restart is required. Package-conflict warnings for unrelated preinstalled apps may still appear; the fresh training interpreter verifies the actual training imports before launch.


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import json
import os
import subprocess
import signal
import sys

# Use PyTorch; avoid importing optional TensorFlow/Flax stacks.
os.environ.setdefault("USE_TF", "0")
os.environ.setdefault("USE_FLAX", "0")

REPO_URL = "https://github.com/rquq/GECC_VLSP_26.git"
BRANCH = "kaggle-linux"
PROJECT_SUBDIR = "."  # The project is at repository root.
CLASSIFICATION_CONFIG = "configs/baseline.json"
CLASSIFICATION_OVERRIDES = {}  # Optional baseline experiment overrides.
CORRECTION_CONFIG = "configs/correction.json"
CORRECTION_OVERRIDES = {}  # Optional Task 2 experiment overrides.
RUN_MODE = "both"  # "task1", "task2" (requires prior classifier), or "both".
RUN_TRAINING = True  # False runs saved-adapter inference and/or enabled diagnostics.
GENERATE_OOF = True  # Task 1: retain leakage-safe hints for later Task 2 runs.
PRIOR_RUN = None  # e.g. "/kaggle/input/my-private-prev-version/runs/RUN"
CLASSIFIER_RUN = None  # Optional direct classifier run; overrides prior classifier.
RESUME_CORRECTION = False  # Continue the prior corrector with a NEW optimizer.
INITIAL_ADAPTER = None  # Optional explicit compatible correction adapter.
EDIT_CORRECTION = False  # Fresh adapter: localized character edits instead of a sentence.
HINT_ABLATION = False  # Paired saved-adapter predicted-hint versus no-hint evaluation.
CANDIDATE_RERANKING = False  # Beam candidates + unchanged source, real quality scoring.
EXPERIMENT_RUN = None  # Defaults to this run, or PRIOR_RUN if RUN_TRAINING=False.
ORIGINAL_DATA = None  # e.g. "/kaggle/input/my-private-dataset/VGEC 2026_v1.json"
CLASSIFIER_DATA = None  # e.g. "/kaggle/input/my-private-dataset/VGEC_classifier.json"
CORRECTION_DATA = None  # e.g. "/kaggle/input/my-private-dataset/VGEC_balanced.json"
SUBMISSION_DATA = "auto"  # Discover attached public_test.json; explicit /kaggle/input path overrides; None uses validation.
USE_HF_SECRET = False
TRACKING_ENABLED = True
WANDB_PROJECT = "vlsp26-gec"
WANDB_ENTITY = None  # Optional team/account; None uses the authenticated default.
WANDB_MODE = os.environ.get("WANDB_MODE", "online")  # online, offline, or disabled.
WANDB_LOG_MODELS = True
WANDB_LOG_TEXT = False  # Must stay false: the corpus agreement prohibits redistribution.
WANDB_GROUP = os.environ.get("WANDB_RUN_GROUP")  # None groups all jobs by this output run.
DIAGNOSTICS_DEVICE = "cuda"
SOURCE_ROOT = Path("/kaggle/working")
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S-%f')
print("Branch:", BRANCH, "| configs:", CLASSIFICATION_CONFIG, CORRECTION_CONFIG)



In [ ]:
# Git runs as a subprocess; notebook execution needs no kernel restart.
# Always clone into a fresh directory; old source edits and outputs are untouched.
from urllib.parse import urlsplit
from contextlib import contextmanager
import tempfile
import os

@contextmanager
def github_git():
    env = os.environ.copy()
    env["GIT_TERMINAL_PROMPT"] = "0"
    token = None
    with tempfile.TemporaryDirectory() as temporary:
        if urlsplit(REPO_URL).hostname == "github.com":
            from kaggle_secrets import UserSecretsClient
            try:
                token = UserSecretsClient().get_secret("GITHUB_TOKEN")
            except Exception:
                raise RuntimeError("Add GITHUB_TOKEN in Kaggle Secrets and enable notebook access. "
                                   "The private repository requires Contents: Read-only permission.") from None
            if not token:
                raise RuntimeError("GITHUB_TOKEN is empty; add a repository-read token in Kaggle Secrets.")
            askpass = Path(temporary) / "git-askpass.sh"
            askpass.write_text('#!/bin/sh\ncase "$1" in\n*Username*) printf "%s\\n" x-access-token ;;\n'
                               '*) printf "%s\\n" "$GECC_GITHUB_TOKEN" ;;\nesac\n', encoding="utf-8")
            askpass.chmod(0o700)
            env.update(GIT_ASKPASS=str(askpass), GECC_GITHUB_TOKEN=token)

        def run(*args, cwd=None, check=True):
            result = subprocess.run(["git", "-c", "credential.helper=", *args], cwd=cwd,
                                    env=env, capture_output=True, text=True)
            if token:
                result.stderr = result.stderr.replace(token, "[REDACTED]")
                result.stdout = result.stdout.replace(token, "[REDACTED]")
            if check and result.returncode:
                detail = result.stderr.strip()
                raise RuntimeError("Git repository access failed. Check Kaggle Internet and GITHUB_TOKEN "
                                   "repository access (Contents: Read-only). " + detail)
            return result

        try:
            yield run
        finally:
            env.pop("GECC_GITHUB_TOKEN", None)
            token = None

if urlsplit(REPO_URL).username is not None:
    raise ValueError("Do not embed GitHub credentials in REPO_URL.")

with github_git() as git:
    remote = git("ls-remote", "--exit-code", "--heads", "--", REPO_URL,
                 f"refs/heads/{BRANCH}", check=False)
    if remote.returncode == 2:
        raise RuntimeError(f"Branch {BRANCH!r} does not exist on GitHub. Push it or select a published BRANCH.")
    if remote.returncode != 0:
        raise RuntimeError("Cannot read the Git repository. Enable Kaggle Internet and grant GITHUB_TOKEN "
                           "access to this private repository with Contents: Read-only permission. "
                           f"Git exit code {remote.returncode}: {remote.stderr.strip()}")
    source_dir = Path(tempfile.mkdtemp(prefix="gecc-two-phase-source-", dir=SOURCE_ROOT))
    git("clone", "--depth", "1", "--single-branch", "--branch", BRANCH,
        "--", REPO_URL, str(source_dir))
PROJECT_DIR = (source_dir / PROJECT_SUBDIR).resolve()
if not PROJECT_DIR.is_relative_to(source_dir.resolve()):
    raise ValueError("PROJECT_SUBDIR must stay inside the cloned repository.")
for filename in ["requirements-kaggle.txt", "model/classification.py", "model/correction.py",
                 "model/two_phase.py", CLASSIFICATION_CONFIG, CORRECTION_CONFIG]:
    if not (PROJECT_DIR / filename).is_file():
        raise FileNotFoundError(f"Missing {filename} in {PROJECT_DIR}; check the selected branch and PROJECT_SUBDIR.")
SOURCE_INFO = {"repo_url": REPO_URL, "branch": BRANCH, "project_subdir": PROJECT_SUBDIR,
               "commit": subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=source_dir, text=True).strip()}
print(json.dumps(SOURCE_INFO, indent=2))



In [ ]:
# Keep the notebook kernel untouched: Save Version cannot pause for a restart.
# Reuse Kaggle's CUDA torch through system-site-packages; install training pins locally.
from importlib.metadata import version
import tempfile

installed_torch = version("torch")
TRAINING_ENV = Path(tempfile.mkdtemp(prefix="gecc-training-env-"))
# Kaggle's system Python cannot reliably bootstrap pip through ensurepip.
# Create only the interpreter; the existing host pip manages this environment.
subprocess.run([sys.executable, "-m", "venv", "--without-pip", "--system-site-packages",
                str(TRAINING_ENV)], check=True)
TRAINING_PYTHON = str(TRAINING_ENV / "bin/python")
constraints = TRAINING_ENV / "constraints.txt"
constraints.write_text(f"torch=={installed_torch}\n", encoding="utf-8")
subprocess.run([sys.executable, "-m", "pip", "--python", TRAINING_PYTHON, "install",
                "-r", str(PROJECT_DIR / "requirements-kaggle.txt"),
                "--constraint", str(constraints)], cwd=PROJECT_DIR, check=True)
subprocess.run([TRAINING_PYTHON, "-c",
                "import numpy, scipy, sklearn, torch, transformers, peft, accelerate, bitsandbytes, sentence_transformers, wandb; "
                "from transformers import AutoModelForCausalLM; from sklearn.metrics import roc_curve; "
                f"assert torch.__version__ == {installed_torch!r}; "
                "print('Training imports passed; torch', torch.__version__, 'CUDA', torch.version.cuda, 'W&B', wandb.__version__)"], check=True)
print("Isolated training environment ready; no kernel restart needed.")



In [ ]:
if USE_HF_SECRET:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

TRACKING_CONFIG = {"enabled": TRACKING_ENABLED, "project": WANDB_PROJECT,
                   "entity": WANDB_ENTITY, "mode": WANDB_MODE,
                   "log_models": WANDB_LOG_MODELS, "log_text": WANDB_LOG_TEXT}
if TRACKING_ENABLED and WANDB_MODE == "online" and not os.environ.get("WANDB_API_KEY"):
    from kaggle_secrets import UserSecretsClient
    try:
        os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
    except Exception:
        raise RuntimeError("Add WANDB_API_KEY in Kaggle Secrets and enable notebook access, "
                           "or select offline/disabled tracking.") from None
    if not os.environ["WANDB_API_KEY"]:
        raise RuntimeError("WANDB_API_KEY is empty; update the Kaggle Secret or select offline tracking.")

def apply_overrides(target, overrides):
    for key, value in overrides.items():
        if isinstance(value, dict) and isinstance(target.get(key), dict):
            apply_overrides(target[key], value)
        else:
            target[key] = value

sys.path.insert(0, str(PROJECT_DIR))
from model.runtime import load_config, resolve_data, resolve_submission_data
from model.run_reuse import resolve_run

if RUN_MODE not in ("task1", "task2", "both"):
    raise ValueError("RUN_MODE must be task1, task2, or both")
if RESUME_CORRECTION and INITIAL_ADAPTER:
    raise ValueError("Choose RESUME_CORRECTION or INITIAL_ADAPTER, not both")
if RESUME_CORRECTION and not PRIOR_RUN:
    raise ValueError("RESUME_CORRECTION requires PRIOR_RUN")
if not (RUN_TRAINING and RUN_MODE == "task1" and SUBMISSION_DATA == "auto"):
    submission_path = resolve_submission_data(SUBMISSION_DATA, input_root=Path("/kaggle/input"))
    SUBMISSION_DATA = str(submission_path) if submission_path is not None else None
else:
    SUBMISSION_DATA = None
print("Submission input:", SUBMISSION_DATA or "validation after training (no public test selected)")
if not RUN_TRAINING and not (SUBMISSION_DATA or HINT_ABLATION or CANDIDATE_RERANKING):
    raise ValueError("With RUN_TRAINING=False, set SUBMISSION_DATA or enable a correction diagnostic")
if RUN_TRAINING and RUN_MODE == "task1" and (HINT_ABLATION or CANDIDATE_RERANKING):
    raise ValueError("Task 1 has no corrector; select both/task2 or saved-run experiments")
if RUN_TRAINING and RUN_MODE == "task1" and SUBMISSION_DATA:
    raise ValueError("A full submission requires Task 2; select both/task2 or a saved two-stage run")

def mounted_input(value):
    path = Path(value).expanduser().resolve()
    if not path.is_relative_to(Path("/kaggle/input")):
        raise ValueError(f"Reuse/data input must be attached under /kaggle/input: {path}")
    if not path.exists():
        raise FileNotFoundError(path)
    return path

for value in (PRIOR_RUN, CLASSIFIER_RUN, INITIAL_ADAPTER, EXPERIMENT_RUN):
    if value:
        mounted_input(value)
if RESUME_CORRECTION:
    INITIAL_ADAPTER = str(resolve_run(mounted_input(PRIOR_RUN), stage="correction") / "adapter")
if SUBMISSION_DATA:
    SUBMISSION_DATA = str(mounted_input(SUBMISSION_DATA))
# Save Version retains runs directly; no output-path symlink is required.
PERSISTED_OUTPUT = Path("/kaggle/working/runs")
PERSISTED_OUTPUT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT = PERSISTED_OUTPUT
OUTPUT_DIR = OUTPUT_ROOT / f"sailor-7b-{RUN_MODE}-{RUN_ID}"
os.environ["WANDB_RUN_GROUP"] = WANDB_GROUP or OUTPUT_DIR.name
TRACKING_CONFIG_PATH = PERSISTED_OUTPUT / f"tracking-config-{RUN_ID}.json"
TRACKING_CONFIG_PATH.write_text(json.dumps({"tracking": TRACKING_CONFIG}, indent=2), encoding="utf-8")
command = None
if RUN_TRAINING:
    config = json.loads((PROJECT_DIR / CLASSIFICATION_CONFIG).read_text(encoding="utf-8"))
    correction_config = json.loads((PROJECT_DIR / CORRECTION_CONFIG).read_text(encoding="utf-8"))
    config["oof"]["enabled"] = GENERATE_OOF
    apply_overrides(config, CLASSIFICATION_OVERRIDES)
    apply_overrides(correction_config, CORRECTION_OVERRIDES)
    if EDIT_CORRECTION:
        correction_config["inference"]["output_format"] = "edits"
    config["data"].update(path=ORIGINAL_DATA, augmentation_path=CLASSIFIER_DATA,
                          correction_path=CORRECTION_DATA, input_root="/kaggle/input")
    correction_config["data"].update(path=CORRECTION_DATA, input_root="/kaggle/input")
    for stage in (config, correction_config):
        stage["run"]["output_root"] = str(OUTPUT_ROOT)
        stage["tracking"] = dict(TRACKING_CONFIG)
    fresh_classifier = RUN_MODE != "task2" and not (CLASSIFIER_RUN or PRIOR_RUN)
    needs_correction = RUN_MODE != "task1" or (fresh_classifier and config["oof"]["enabled"])
    if fresh_classifier:
        original_path = mounted_input(resolve_data(config))
        mixed_config = json.loads(json.dumps(config))
        mixed_config["data"].update(path=config["data"]["augmentation_path"],
                                   filename=config["data"]["augmentation_filename"])
        classifier_path = mounted_input(resolve_data(mixed_config))
        config["data"].update(path=str(original_path), augmentation_path=str(classifier_path))
    if needs_correction:
        correction_path = mounted_input(resolve_data(correction_config))
        config["data"]["correction_path"] = str(correction_path)
        correction_config["data"]["path"] = str(correction_path)
    CONFIG_PATH = PERSISTED_OUTPUT / f"classification-config-{RUN_ID}.json"
    CORRECTION_CONFIG_PATH = PERSISTED_OUTPUT / f"correction-config-{RUN_ID}.json"
    for path, stage in ((CONFIG_PATH, config), (CORRECTION_CONFIG_PATH, correction_config)):
        path.write_text(json.dumps(stage, ensure_ascii=False, indent=2), encoding="utf-8")
    load_config(CONFIG_PATH, "classification")
    load_config(CORRECTION_CONFIG_PATH, "correction")
    command = [TRAINING_PYTHON, "-m", "model.two_phase", "--mode", RUN_MODE,
               "--classification-config", str(CONFIG_PATH), "--correction-config", str(CORRECTION_CONFIG_PATH),
               "--output-dir", str(OUTPUT_DIR)]
    if SUBMISSION_DATA:
        command.extend(["--submission-data", SUBMISSION_DATA])
    reuse_flag = "--classifier-run" if CLASSIFIER_RUN else "--prior-run"
    for flag, value in ((reuse_flag, CLASSIFIER_RUN or PRIOR_RUN), ("--initial-adapter", INITIAL_ADAPTER)):
        if value:
            command.extend([flag, str(value)])
    subprocess.run(command + ["--preflight-only"], cwd=PROJECT_DIR, check=True)
elif not (EXPERIMENT_RUN or PRIOR_RUN):
    raise ValueError("Saved inference/diagnostics require EXPERIMENT_RUN or PRIOR_RUN")


In [ ]:
def redact(text):
    for name in ("WANDB_API_KEY", "HF_TOKEN", "HUGGING_FACE_HUB_TOKEN"):
        secret = os.environ.get(name)
        if secret:
            text = text.replace(secret, "[REDACTED]")
    return text

def stream_command(arguments, log_path):
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open("w", encoding="utf-8") as log_file:
        log_file.write(json.dumps(SOURCE_INFO, ensure_ascii=False) + "\n")
        log_file.flush()
        with subprocess.Popen(arguments, cwd=PROJECT_DIR, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1,
                              start_new_session=True) as process:
            try:
                for line in process.stdout:
                    line = redact(line)
                    print(line, end="", flush=True)
                    log_file.write(line)
                    log_file.flush()
                returncode = process.wait()
            except BaseException as error:
                try:
                    os.killpg(process.pid, signal.SIGINT if isinstance(error, KeyboardInterrupt) else signal.SIGTERM)
                except ProcessLookupError:
                    pass
                try:
                    process.wait(timeout=30)
                except subprocess.TimeoutExpired:
                    try:
                        os.killpg(process.pid, signal.SIGKILL)
                    except ProcessLookupError:
                        pass
                    process.wait()
                raise
        if returncode:
            raise subprocess.CalledProcessError(returncode, arguments)

LOG_PATH = OUTPUT_DIR.parent / f"{OUTPUT_DIR.name}.log"
if RUN_TRAINING:
    stream_command(command, LOG_PATH)
if not RUN_TRAINING and SUBMISSION_DATA:
    inference_command = [TRAINING_PYTHON, "-m", "model.pipeline_predict",
                         "--run", EXPERIMENT_RUN or PRIOR_RUN, "--data", SUBMISSION_DATA,
                         "--output", str(OUTPUT_DIR / "submission.json"), "--device", DIAGNOSTICS_DEVICE]
    stream_command(inference_command, LOG_PATH)
EXPERIMENT_OUTPUT = None
if HINT_ABLATION or CANDIDATE_RERANKING:
    experiment_run = EXPERIMENT_RUN or (str(OUTPUT_DIR) if RUN_TRAINING else PRIOR_RUN)
    EXPERIMENT_OUTPUT = OUTPUT_ROOT / f"correction-experiments-{RUN_ID}"
    experiment_command = [TRAINING_PYTHON, "-m", "model.correction_experiments",
                          "--run", str(experiment_run), "--output-dir", str(EXPERIMENT_OUTPUT),
                          "--device", DIAGNOSTICS_DEVICE, "--tracking-config", str(TRACKING_CONFIG_PATH)]
    if HINT_ABLATION:
        experiment_command.append("--hint-ablation")
    if CANDIDATE_RERANKING:
        experiment_command.append("--rerank")
    stream_command(experiment_command, EXPERIMENT_OUTPUT.with_suffix(".log"))


In [ ]:
from IPython.display import JSON, FileLink, display
import shutil
if RUN_TRAINING:
    for filename in ["classification/metrics.json", "correction/metrics.json", "submission.metrics.json", "pipeline_manifest.json", "run_handoff.json"]:
        path = OUTPUT_DIR / filename
        if path.is_file():
            print(filename)
            display(JSON(json.loads(path.read_text(encoding="utf-8"))))
    ARCHIVE_PATH = Path(shutil.make_archive(str(PERSISTED_OUTPUT / OUTPUT_DIR.name) + "-artifacts", "zip", OUTPUT_DIR))
    display(FileLink(str(ARCHIVE_PATH)))
    display(FileLink(str(LOG_PATH.resolve())))
    print("Next run: attach this private saved output and set PRIOR_RUN to its mounted /kaggle/input/... run folder, not", OUTPUT_DIR)
if (OUTPUT_DIR / "submission.json").is_file():
    display(FileLink(str(OUTPUT_DIR / "submission.json")))
    print("Submission scope:", json.loads((OUTPUT_DIR / "submission.evaluation.json").read_text(encoding="utf-8")))
if EXPERIMENT_OUTPUT:
    print("Paired correction diagnostics:", EXPERIMENT_OUTPUT)
    for path in sorted(EXPERIMENT_OUTPUT.rglob("*metrics*.json")):
        print(path.relative_to(EXPERIMENT_OUTPUT))
        display(JSON(json.loads(path.read_text(encoding="utf-8"))))
print("Save a PRIVATE notebook version to retain /kaggle/working/runs.")


## Saved pipeline use

Each training mode writes `run_handoff.json` with relative stage paths. Reused classifier weights, tokenizers, OOF hints and provenance are copied into the new run; the next handoff does not depend on an older input mount. Validation scores reuse the selection holdout and are exploratory, not an independent test score. Logs, ZIPs and run folders remain under `/kaggle/working/runs`.

**Next Save Version:** save this notebook privately, then attach its saved output through Add Input, within the agreement's authorized access. Set `PRIOR_RUN` to the attached `/kaggle/input/.../runs/RUN` and `RUN_MODE="task2"` to reuse compatible Task 1/OOF artifacts. Changed datasets or model bases require fresh compatible fits. Attachments are manual; a mount containing multiple runs needs an explicit path.

Set `RESUME_CORRECTION=True` to continue a compatible prior Task 2 adapter with a fresh optimizer/scheduler. Default is a fresh Task 2 fit, even when Task 1 is reused. Set `INITIAL_ADAPTER` for an explicit corrector instead. Model revision, LoRA layout, hint mode and sentence/edit representation must match; a sentence adapter cannot be silently converted to an edit adapter. No arbitrary incompatible checkpoint conversion. Changed data invalidates old OOF coverage/provenance: retrain Task 1 and its OOF fits on the new audited build.

`EDIT_CORRECTION=True` trains JSON character edits (`start`, `end`, `replacement`) against the original Unicode string; untouched spans are retained deterministically. Invalid edits remain errors. `CANDIDATE_RERANKING=True` compares beam outputs and the unchanged source with a frozen-base grammaticality judgment and a CPU multilingual semantic encoder, using edit size only on tied quality scores. It adds inference/download cost and does not automatically favor the unchanged source. `HINT_ABLATION=True` compares the same saved adapter with predicted hints and no label/type hint; the no-hint prompt is a diagnostic distribution shift.

For public-test inference with saved adapters, set `RUN_TRAINING=False`, `PRIOR_RUN` and leave `SUBMISSION_DATA="auto"` (or set an explicit `/kaggle/input/.../public_test.json` path). Both adapters are required. Alternatively:

```bash
python -m model.pipeline_predict --run /kaggle/working/runs/RUN \
  --data /kaggle/input/DATA/public_test.json --output /kaggle/working/runs/new-submission/submission.json
```

`submission.json` contains only organizer `ID` / `Output` fields. Invalid/blank corrections fail export; comparisons and any labeled metrics remain separate private artifacts. Combined scores are a non-official equal-weight mean of macro-F1 and error-only GLEU+; selection is unchanged.

```bash
python -m venv --without-pip --system-site-packages /tmp/gecc-training
python -m pip --python /tmp/gecc-training/bin/python install -r requirements-kaggle.txt
/tmp/gecc-training/bin/python -m model.two_phase --classification-config configs/baseline.json \
  --correction-config configs/correction.json --output-dir /kaggle/working/runs/RUN
```
